# Policy Translation

This notebook consumes model outputs from `model.ipynb` (no legacy part files) and converts them into actionable policy insights.

**Inputs (from `model_output/`):**
- `model_a_metrics.csv`
- `modelb_results.csv`
- `modelc_results.csv`
- `modelb_shap_importance.csv`
- `modelc_coefficients_sorted.csv`
- `modelb_modelc_feature_comparison.csv`

**Outputs (to `model_output/`):**
- `policy_model_comparison.csv`
- `policy_feature_signals.csv`
- `policy_actions.csv`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
print("Libraries loaded")

In [ ]:
ROOT = Path(".")
OUT = ROOT / "model_output"

A_METRIC = OUT / "model_a_metrics.csv"
B_METRIC = OUT / "modelb_results.csv"
C_METRIC = OUT / "modelc_results.csv"
SHAP_IN = OUT / "modelb_shap_importance.csv"
COEF_IN = OUT / "modelc_coefficients_sorted.csv"
CMP_IN = OUT / "modelb_modelc_feature_comparison.csv"

required = [A_METRIC, B_METRIC, C_METRIC, SHAP_IN, COEF_IN, CMP_IN]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required model output files:\n" + "\n".join(missing))

metrics_a = pd.read_csv(A_METRIC)
metrics_b = pd.read_csv(B_METRIC)
metrics_c = pd.read_csv(C_METRIC)
shap_imp = pd.read_csv(SHAP_IN)
coef_sorted = pd.read_csv(COEF_IN)
cmp_bc = pd.read_csv(CMP_IN)

print("Loaded files:")
print("metrics_a:", metrics_a.shape)
print("metrics_b:", metrics_b.shape)
print("metrics_c:", metrics_c.shape)
print("shap_imp:", shap_imp.shape)
print("coef_sorted:", coef_sorted.shape)
print("cmp_bc:", cmp_bc.shape)

## Model Performance Summary

In [ ]:
metrics_all = pd.concat([metrics_a, metrics_b, metrics_c], ignore_index=True)
metrics_all["model"] = metrics_all["model"].replace({
    "RF_A": "Model_A_RF",
    "XGBoost_A": "Model_A_XGBoost",
    "RandomForest_B": "Model_B_RF",
    "Ridge_C": "Model_C_Ridge",
})

avg_by_model = (
    metrics_all.groupby("model", as_index=False)
    .agg(avg_RMSE=("RMSE", "mean"), avg_MAE=("MAE", "mean"), avg_R2=("R2", "mean"))
    .sort_values(["avg_RMSE", "avg_MAE", "avg_R2"], ascending=[True, True, False])
)

champion = avg_by_model.iloc[0]["model"]
print("Champion model by avg RMSE:", champion)
display(avg_by_model)

avg_by_model.to_csv(OUT / "policy_model_comparison.csv", index=False)
print("Saved:", OUT / "policy_model_comparison.csv")

## Feature Signals for Policy

In [ ]:
signals = cmp_bc.copy()
signals["cross_model_strength"] = signals["mean_abs_shap"] * signals["abs_ridge_coef"]
signals = signals.sort_values(["cross_model_strength", "mean_abs_shap"], ascending=False)

policy_signals = signals.head(20).copy()
display(policy_signals)

policy_signals.to_csv(OUT / "policy_feature_signals.csv", index=False)
print("Saved:", OUT / "policy_feature_signals.csv")

## Proposed Actions

This table is a lightweight template to convert top quantitative signals into policy-facing recommendations.

In [ ]:
def action_for_feature(f):
    f_low = str(f).lower()
    if "green_ratio" in f_low:
        return "Increase urban greening and tree canopy in dense neighborhoods"
    if "wind" in f_low:
        return "Use ventilation corridors in urban planning"
    if "precipitation" in f_low:
        return "Plan seasonal mitigation for dry periods"
    if "temp" in f_low or "month" in f_low:
        return "Target seasonal campaigns and winter anti-inversion policies"
    if "no2" in f_low or "pm10" in f_low:
        return "Prioritize traffic/combustion emission reduction in hotspots"
    if "pm2_5_lag" in f_low or "roll3" in f_low:
        return "Deploy early warning + short-term interventions when persistence is high"
    if "latitude" in f_low or "longitude" in f_low or "altitude" in f_low:
        return "Use place-based local plans by geography"
    return "Investigate targeted local intervention"

actions = policy_signals[["feature", "mean_abs_shap", "ridge_coef", "ridge_direction"]].copy()
actions["recommended_action"] = actions["feature"].apply(action_for_feature)
display(actions.head(15))

actions.to_csv(OUT / "policy_actions.csv", index=False)
print("Saved:", OUT / "policy_actions.csv")